In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/ngocccchannn/full-interactions/interactions_full.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/train.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_5/test_target.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_5/test_fold_in.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_5/val_fold_in.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_5/val_target.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_1/test_target.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_1/test_fold_in.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_1/val_fold_in.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_1/val_target.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_3/test_target.parquet
/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_3/test_fold_in.parquet
/kaggle/input/da

In [2]:
import os
import shutil
import json
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import torch
from IPython.display import FileLink

print("Đã khởi tạo xong môi trường!")

Đã khởi tạo xong môi trường!


In [3]:
# A. Kịch bản Warm
tr_warm = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/train_warm.parquet')
va_warm = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/val_warm.parquet')
te_warm = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/warm-scenario/test_warm.parquet')

# B. Kịch bản Cold-item
tr_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/col-scenario-data/train_cold.parquet')
va_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/col-scenario-data/val_cold.parquet')
te_cold = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/col-scenario-data/test_cold.parquet')

with open("/kaggle/input/datasets/ngocccchannn/cold-scenario/cold_items_test.json", "r", encoding="utf-8") as f:
    cold_test = set(json.load(f))

# C. Kịch bản Few-shot-user (k = 1, 3, 5)
train_fs = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/train.parquet')

few_shot_data = {}
for k in [1, 3, 5]:
    base_path = f'/kaggle/input/datasets/ngocccchannn/few-shot-user-scenario/k_{k}'
    few_shot_data[k] = {
        'val_fold_in': pd.read_parquet(f'{base_path}/val_fold_in.parquet'),
        'val_target': pd.read_parquet(f'{base_path}/val_target.parquet'),
        'test_fold_in': pd.read_parquet(f'{base_path}/test_fold_in.parquet'),
        'test_target': pd.read_parquet(f'{base_path}/test_target.parquet')
    }

# D. Dữ liệu tương tác tổng thể
inter = pd.read_parquet('/kaggle/input/datasets/ngocccchannn/full-interactions/interactions_full.parquet')

print("Đã nạp thành công toàn bộ dữ liệu cho các kịch bản!")

Đã nạp thành công toàn bộ dữ liệu cho các kịch bản!


In [4]:
@torch.no_grad() 
def evaluate(scores_fn, target, mask, ks=(5, 10, 20), cand=None, bs=1024, tie_seed=0): 
    users = sorted(target); K = max(ks) 
    out = {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks} 
    g = torch.Generator().manual_seed(tie_seed) 
    disc = 1.0 / np.log2(np.arange(2, K + 2)) 
    for s in range(0, len(users), bs): 
        ub = users[s:s + bs]; sc = scores_fn(torch.tensor(ub)).float().cpu() 
        if cand is not None: sc[:, ~torch.as_tensor(cand)] = -float("inf") 
        for r, u in enumerate(ub): 
            if mask.get(u): sc[r, list(mask[u])] = -float("inf") 
        perm = torch.randperm(sc.size(1), generator=g)     # hoán vị ngẫu nhiên thứ tự khóa 
        order = torch.sort(sc[:, perm], dim=1, descending=True, stable=True).indices 
        top = perm[order[:, :K]].numpy()                    # khi hòa điểm: thứ tự ngẫu nhiên 
        for r, u in enumerate(ub): 
            gt = target[u]; hit = np.isin(top[r], list(gt)).astype(float) 
            for k in ks: 
                h = hit[:k]; idcg = disc[:min(len(gt), k)].sum() 
                out[f"Recall@{k}"].append(h.sum() / len(gt)) 
                out[f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg) 
                out[f"HR@{k}"].append(float(h.sum() > 0)) 
    return {m: np.array(v) for m, v in out.items()} 

In [5]:
def evaluate_and_save(scores_fn, target, mask, path=None, **kw):
    """Chạy evaluate và lưu điểm từng học viên ra file parquet"""
    res = evaluate(scores_fn, target, mask, **kw)
    if path is not None:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        table = pd.DataFrame(res)
        table.insert(0, 'user_idx', sorted(target))
        table.to_parquet(path, index=False)
    return res

In [6]:
# Thiết lập cấu hình chung
SEEDS = (42, 2023, 2024, 1007, 999)
KS = (5, 10, 20)
OUT_DIR = '/kaggle/working/eval_results'

all_courses = sorted(inter['course_id'].unique())
item2idx = {c: i for i, c in enumerate(all_courses)}
n_items = len(all_courses)

# Tập ứng viên (a): Chỉ 502 khóa học cold-test 
cand_a = np.zeros(n_items, dtype=bool)
cand_a[[item2idx[c] for c in cold_test]] = True
assert cand_a.sum() == len(cold_test) == 502, 'Tập ứng viên (a) phải đủ 502 khóa cold-test'

# Tập ứng viên (b): Toàn bộ khóa học 
cand_b = np.ones(n_items, dtype=bool)

def history_dict(d, user2idx):
    out = {}
    u = d['user_id'].map(user2idx)
    i = d['course_id'].map(item2idx)
    ok = u.notna() & i.notna()
    for uu, ii in zip(u[ok].astype(int), i[ok].astype(int)):
        out.setdefault(uu, set()).add(ii)
    return out

def pop_vector(d):
    counts = d['course_id'].map(item2idx).dropna().astype(int).value_counts()
    vec = np.zeros(n_items, dtype=np.float32)
    if len(counts):
        vec[counts.index.to_numpy()] = (counts.values / counts.max()).astype(np.float32)
    return vec

In [7]:
# Test 1 - One-hot đúng item cho Recall/ NDCG = 1
def run_evaluator_test_1():
    print("TEST 1: Điểm chuẩn xác tuyệt đối")
    
    n_items = 100
    target = {0: {15}}
    mask = {0: {5, 6, 7}}

    def scores_onehot(users):
        sc = torch.full((len(users), n_items), -10.0, dtype=torch.float32)
        sc[:, 15] = 100.0
        return sc

    res = evaluate(scores_onehot, target, mask, ks=(5, 10, 20))
    for k in (5, 10, 20):
        assert res[f"Recall@{k}"][0] == 1.0 and abs(res[f"NDCG@{k}"][0] - 1.0) < 1e-12, f"Test 1 không đạt ở k={k}"
    
    # Kiểm tra khóa đúng đứng thứ 2
    def scores_rank2(users):
        sc = torch.full((len(users), n_items), -10.0, dtype=torch.float32)
        sc[:, 40], sc[:, 15] = 100.0, 50.0
        return sc
    res2 = evaluate(scores_rank2, target, mask, ks=(5,))
    assert abs(res2["NDCG@5"][0] - 1.0 / np.log2(3)) < 1e-9, "NDCG khi khóa đúng đứng thứ 2 không khớp"
    
    # Kiểm tra khóa bị che không được tính
    def scores_masked(users):
        sc = torch.full((len(users), n_items), -10.0, dtype=torch.float32)
        sc[:, 5] = 100.0
        return sc
    res3 = evaluate(scores_masked, {0: {5}}, {0: {5}}, ks=(5,))
    assert res3["Recall@5"][0] == 0.0, "Khóa bị che vẫn bị tính điểm"
    print("[Test 1 ĐẠT] Recall = NDCG = 1.0 ở k = 5, 10, 20 và xử lý đúng mask/rank phụ.")

run_evaluator_test_1()

TEST 1: Điểm chuẩn xác tuyệt đối
[Test 1 ĐẠT] Recall = NDCG = 1.0 ở k = 5, 10, 20 và xử lý đúng mask/rank phụ.


In [8]:
# Test 2 - Xếp hạng ngẫu nhiên khớp kỳ vọng thống kê 
def run_evaluator_test_2():
    print("TEST 2: Kiểm tra phân phối ngẫu nhiên")

    n_items = 100
    rng = np.random.default_rng(42)
    n_users, n_mask = 4000, 3
    target2, mask2 = {}, {}
    for u in range(n_users):
        chosen = rng.choice(n_items, size=n_mask + 1, replace=False)
        mask2[u], target2[u] = set(chosen[:n_mask].tolist()), {int(chosen[n_mask])}
    n_cand = n_items - n_mask

    score_rng = np.random.default_rng(2026)
    def scores_random(users):
        return torch.tensor(score_rng.random((len(users), n_items)), dtype=torch.float32)

    res = evaluate(scores_random, target2, mask2, ks=(5, 10, 20))
    for k in (5, 10, 20):
        expected = k / n_cand
        actual = res[f"Recall@{k}"].mean()
        tol = 4 * np.sqrt(expected * (1 - expected) / n_users)
        print(f"[Test 2] Recall@{k} thực tế = {actual:.4f} | kỳ vọng = {expected:.4f} | sai lệch cho phép = {tol:.4f}")
        assert abs(actual - expected) <= tol, f"Test 2 không đạt ở k={k}"
    print("[Test 2 ĐẠT] Kết quả ngẫu nhiên khớp kỳ vọng thống kê.")

run_evaluator_test_2()

TEST 2: Kiểm tra phân phối ngẫu nhiên
[Test 2] Recall@5 thực tế = 0.0488 | kỳ vọng = 0.0515 | sai lệch cho phép = 0.0140
[Test 2] Recall@10 thực tế = 0.0998 | kỳ vọng = 0.1031 | sai lệch cho phép = 0.0192
[Test 2] Recall@20 thực tế = 0.2018 | kỳ vọng = 0.2062 | sai lệch cho phép = 0.0256
[Test 2 ĐẠT] Kết quả ngẫu nhiên khớp kỳ vọng thống kê.


In [9]:
# Test 3: điểm hằng trên tập (a) -> Recall@20 xấp xỉ 20 / 502 
def run_evaluator_test_3(target, mask, cand_a):
    print(" CHẠY TEST 3: Kiểm tra điểm hằng trên tập ứng viên (a)")

    # Điểm hằng trên tập ứng viên (a): Recall@20 xấp xỉ 20/ 502 
    res_const = evaluate(lambda u: torch.zeros(len(u), n_items), target, mask, cand=cand_a, ks=(20,))
    actual_r20 = res_const["Recall@20"].mean()
    expected_r20 = 20 / 502
    print(f"[Test 3] Recall@20 điểm hằng thực tế = {actual_r20:.4f} | Kỳ vọng lý thuyết = {expected_r20:.4f}")

In [10]:
# Chạy Test 3 kiểm tra điểm hằng trên tập cold-item
users_cold = sorted(set(tr_cold['user_id']) | set(te_cold['user_id']))
user2idx_cold = {u: i for i, u in enumerate(users_cold)}
target_cold = history_dict(te_cold, user2idx_cold)
hist_cold = history_dict(tr_cold, user2idx_cold)
run_evaluator_test_3(target_cold, hist_cold, cand_a)

 CHẠY TEST 3: Kiểm tra điểm hằng trên tập ứng viên (a)
[Test 3] Recall@20 điểm hằng thực tế = 0.0194 | Kỳ vọng lý thuyết = 0.0398


In [11]:
# Sanity check - Kiểm tra xáo trộn nhãn train 
def expected_random_recall(target, mask, cand, k, n_items):
    """Tính Recall kỳ vọng khi xếp ngẫu nhiên dựa trên số ứng viên và mask."""
    n_cand = n_items if cand is None else int(cand.sum())
    vals = []
    for u in target:
        m = mask.get(u, ())
        n_masked = len(m) if cand is None else sum(1 for i in m if cand[i])
        vals.append(min(1.0, k / max(1, n_cand - n_masked)))
    return float(np.mean(vals))

def run_label_shuffle_sanity_check(train_df, test_df, scenario, cand_sets, tol_factor=2.0):
    print(f"SANITY CHECK: MOSTPOP TRÊN NHÃN XÁO | Kịch bản: {scenario.upper()}")
    users = sorted(set(train_df['user_id']) | set(test_df['user_id']))
    user2idx = {u: i for i, u in enumerate(users)}
    target = history_dict(test_df, user2idx)
    hist = history_dict(train_df, user2idx)
    train_courses = train_df['course_id'].unique()

    rows = []
    for tag, cand in cand_sets.items():
        print(f"\n--- Tập ứng viên: {tag} ({int(cand.sum()):,} khóa) ---")
        exp20 = expected_random_recall(target, hist, cand, 20, n_items)
        
        res_shuf = []
        for sd in SEEDS:
            g = np.random.default_rng(sd)
            shuf = train_df.copy()
            shuf['course_id'] = g.choice(train_courses, size=len(shuf))
            h_s = history_dict(shuf, user2idx)
            p_s = pop_vector(shuf)
            sc_s = lambda u, p=p_s: torch.tensor(np.broadcast_to(p, (len(u), n_items)), dtype=torch.float32)
            res_shuf.append(evaluate(sc_s, target, h_s, ks=KS, cand=cand, tie_seed=sd))
        
        r = {'kịch bản': scenario, 'tập ứng viên': tag, 'mô hình': 'MostPop (nhãn train đã xáo)'}
        print(f" {'Metric':<12} | {'@5':<8} | {'@10':<8} | {'@20':<8}")
        print("-" * 45)
        for m in ('Recall', 'NDCG'):
            vals = [np.mean([x[f'{m}@{k}'].mean() for x in res_shuf]) for k in KS]
            for k, val in zip(KS, vals):
                r[f'{m}@{k}'] = float(val)
            print(f" {m:<12} | {vals[0]:<8.4f} | {vals[1]:<8.4f} | {vals[2]:<8.4f}")
        rows.append(r)
        
        shuf20 = r['Recall@20']
        print(f"[Kiểm chứng định lượng] Recall@20 thực tế sau xáo nhãn = {shuf20:.4f} | Kỳ vọng ngẫu nhiên lý thuyết = {exp20:.4f}")
        print("[LƯU Ý] Đây là kết quả của MostPop trên nhãn xáo (phản ánh phân phối độ phổ biến sau xáo), không phải mô hình học được, do đó sẽ chạy kiểm tra rò rỉ chuẩn trên LightGCN.")
    return pd.DataFrame(rows)

In [12]:
# Baseline MostPop 
def run_mostpop_baseline(train_df, test_df, scenario, cand_sets):
    print(f" CHẠY BASELINE: MOSTPOP | Kịch bản: {scenario.upper()}")
    users = sorted(set(train_df['user_id']) | set(test_df['user_id']))
    user2idx = {u: i for i, u in enumerate(users)}
    target = history_dict(test_df, user2idx)
    hist = history_dict(train_df, user2idx)
    pop = pop_vector(train_df)

    rows = []
    for tag, cand in cand_sets.items():
        print(f"\n--- Tập ứng viên: {tag} ({int(cand.sum()):,} khóa) ---")
        sc_pop = lambda u, p=pop: torch.tensor(np.broadcast_to(p, (len(u), n_items)), dtype=torch.float32)
        res_pop = evaluate_and_save(sc_pop, target, hist, path=f'{OUT_DIR}/{scenario}_{tag}_mostpop.parquet', ks=KS, cand=cand, tie_seed=42)
        
        r = {'kịch bản': scenario, 'tập ứng viên': tag, 'mô hình': 'MostPop'}
        print(f" {'Metric':<12} | {'@5':<8} | {'@10':<8} | {'@20':<8}")
        print("-" * 45)
        for m in ('Recall', 'NDCG'):
            vals = [res_pop[f'{m}@{k}'].mean() for k in KS]
            for k, val in zip(KS, vals):
                r[f'{m}@{k}'] = float(val)
            print(f" {m:<12} | {vals[0]:<8.4f} | {vals[1]:<8.4f} | {vals[2]:<8.4f}")
        rows.append(r)
    return pd.DataFrame(rows)

In [13]:
# Baseline Random 
def run_random_baseline(train_df, test_df, scenario, cand_sets):
    print(f" CHẠY BASELINE: RANDOM (5 SEEDS) | Kịch bản: {scenario.upper()}")
    users = sorted(set(train_df['user_id']) | set(test_df['user_id']))
    user2idx = {u: i for i, u in enumerate(users)}
    target = history_dict(test_df, user2idx)
    hist = history_dict(train_df, user2idx)

    rows = []
    for tag, cand in cand_sets.items():
        print(f"\n--- Tập ứng viên: {tag} ({int(cand.sum()):,} khóa) ---")
        res_rand = []
        for sd in SEEDS:
            g = np.random.default_rng(sd)
            sc_rand = lambda u, g=g: torch.tensor(g.random((len(u), n_items)), dtype=torch.float32)
            path = f'{OUT_DIR}/{scenario}_{tag}_random_seed{sd}.parquet' if sd == SEEDS[0] else None
            res_rand.append(evaluate_and_save(sc_rand, target, hist, path=path, ks=KS, cand=cand, tie_seed=sd))
        
        r = {'kịch bản': scenario, 'tập ứng viên': tag, 'mô hình': 'Random'}
        print(f" {'Metric':<12} | {'@5 (Mean ± Std)':<18} | {'@10':<18} | {'@20':<18}")
        print("-" * 72)
        for m in ('Recall', 'NDCG'):
            means = [np.mean([x[f'{m}@{k}'].mean() for x in res_rand]) for k in KS]
            stds = [np.std([x[f'{m}@{k}'].mean() for x in res_rand]) for k in KS]
            for k, val in zip(KS, means):
                r[f'{m}@{k}'] = float(val)
            print(f" {m:<12} | {means[0]:.4f}±{stds[0]:.4f}  | {means[1]:.4f}±{stds[1]:.4f}  | {means[2]:.4f}±{stds[2]:.4f}")
        rows.append(r)
    return pd.DataFrame(rows)

In [14]:
warm_cand_sets = {
    'b_toan_bo_khoa': cand_b
}
cold_cand_sets = {
    'a_chi_khoa_cold_test': cand_a,
    'b_toan_bo_khoa': cand_b
}

In [15]:
print("\n KỊCH BẢN WARM")
run_mostpop_baseline(tr_warm, te_warm, 'warm', warm_cand_sets)
run_random_baseline(tr_warm, te_warm, 'warm', warm_cand_sets)
run_label_shuffle_sanity_check(tr_warm, te_warm, 'warm', warm_cand_sets)


 KỊCH BẢN WARM
 CHẠY BASELINE: MOSTPOP | Kịch bản: WARM

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0219   | 0.0466   | 0.0753  
 NDCG         | 0.0208   | 0.0314   | 0.0410  
 CHẠY BASELINE: RANDOM (5 SEEDS) | Kịch bản: WARM

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5 (Mean ± Std)    | @10                | @20               
------------------------------------------------------------------------
 Recall       | 0.0013±0.0001  | 0.0027±0.0004  | 0.0055±0.0003
 NDCG         | 0.0017±0.0002  | 0.0020±0.0001  | 0.0031±0.0002
SANITY CHECK: MOSTPOP TRÊN NHÃN XÁO | Kịch bản: WARM

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0017   | 0.0066   | 0.0103  
 NDCG         | 0.0017   | 0.0036   | 0.0050  
[Kiểm chứng định lượng] Recall@20 t

,kịch bản,tập ứng viên,mô hình,Recall@5,Recall@10,Recall@20,NDCG@5,NDCG@10,NDCG@20
0,warm,b_toan_bo_khoa,MostPop (nhãn train đã xáo),0.001675,0.006559,0.010273,0.001669,0.003625,0.00503


In [16]:
print("\n KỊCH BẢN COLD-ITEM")
run_mostpop_baseline(tr_cold, te_cold, 'cold-item', cold_cand_sets)
run_random_baseline(tr_cold, te_cold, 'cold-item', cold_cand_sets)
run_label_shuffle_sanity_check(tr_cold, te_cold, 'cold-item', cold_cand_sets)


 KỊCH BẢN COLD-ITEM
 CHẠY BASELINE: MOSTPOP | Kịch bản: COLD-ITEM

--- Tập ứng viên: a_chi_khoa_cold_test (502 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0029   | 0.0061   | 0.0340  
 NDCG         | 0.0029   | 0.0040   | 0.0116  

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0000   | 0.0000   | 0.0000  
 NDCG         | 0.0000   | 0.0000   | 0.0000  
 CHẠY BASELINE: RANDOM (5 SEEDS) | Kịch bản: COLD-ITEM

--- Tập ứng viên: a_chi_khoa_cold_test (502 khóa) ---
 Metric       | @5 (Mean ± Std)    | @10                | @20               
------------------------------------------------------------------------
 Recall       | 0.0094±0.0015  | 0.0195±0.0013  | 0.0405±0.0029
 NDCG         | 0.0062±0.0004  | 0.0098±0.0004  | 0.0157±0.0010

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       

,kịch bản,tập ứng viên,mô hình,Recall@5,Recall@10,Recall@20,NDCG@5,NDCG@10,NDCG@20
0,cold-item,a_chi_khoa_cold_test,MostPop (nhãn train đã xáo),0.010297,0.017902,0.033522,0.007187,0.00984,0.014217
1,cold-item,b_toan_bo_khoa,MostPop (nhãn train đã xáo),0.000000,0.000000,0.000000,0.000000,0.00000,0.000000


In [ ]:
print("\n KỊCH BẢN FEW-SHOT-USER")
for k in [1, 3, 5]:
    print(f"\n--- Cấu hình Few-shot với k = {k} ---")
    te_target = few_shot_data[k]['test_target']
    run_mostpop_baseline(train_fs, te_target, f'few-shot-k{k}', warm_cand_sets)
    run_random_baseline(train_fs, te_target, f'few-shot-k{k}', warm_cand_sets)
    run_label_shuffle_sanity_check(train_fs, te_target, f'few-shot-k{k}', warm_cand_sets)


 KỊCH BẢN FEW-SHOT-USER

--- Cấu hình Few-shot với k = 1 ---
 CHẠY BASELINE: MOSTPOP | Kịch bản: FEW-SHOT-K1

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0241   | 0.0652   | 0.1221  
 NDCG         | 0.0356   | 0.0590   | 0.0833  
 CHẠY BASELINE: RANDOM (5 SEEDS) | Kịch bản: FEW-SHOT-K1

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5 (Mean ± Std)    | @10                | @20               
------------------------------------------------------------------------
 Recall       | 0.0007±0.0002  | 0.0018±0.0004  | 0.0043±0.0007
 NDCG         | 0.0018±0.0003  | 0.0023±0.0004  | 0.0033±0.0005
SANITY CHECK: MOSTPOP TRÊN NHÃN XÁO | Kịch bản: FEW-SHOT-K1

--- Tập ứng viên: b_toan_bo_khoa (3,769 khóa) ---
 Metric       | @5       | @10      | @20     
---------------------------------------------
 Recall       | 0.0014   | 0.0052   | 0.0106  
 NDCG         |

In [18]:
print('Completed!')

Completed!
